In [15]:
#importations des premiers bibliotheques 
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import warnings 
warnings.filterwarnings('ignore')



#chargement du dataset
df = pd.read_excel('Dataset_complet_Meteo.xlsx')

                                    ANALYSE DE FORME(comprendre la structure des donnees)

In [21]:
# taille du dataset 
df.shape



(87240, 26)

In [23]:
# entête du datatset 

df.head()

,id,time,weather_code,temperature_2m_max,temperature_2m_min,temperature_2m_mean,apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,sunrise,...,precipitation_hours,wind_speed_10m_max,wind_gusts_10m_max,wind_direction_10m_dominant,shortwave_radiation_sum,et0_fao_evapotranspiration,city,region,latitude,longitude
0,1,2020-01-01,3,33.2,2026-09-21 00:00:00,2026-05-26 00:00:00,34.0,2026-06-25 00:00:00,2026-01-29 00:00:00,2020-01-01 06:22:00,...,0.0,2026-05-10 00:00:00,2026-05-24 00:00:00,132,20.18,4.59,Bafia,Centre,4.75,11.23
1,2,2020-01-02,3,31.9,2026-09-21 00:00:00,2026-09-25 00:00:00,2026-01-31 00:00:00,2026-03-23 00:00:00,27.0,2020-01-02 06:23:00,...,0.0,2026-05-08 00:00:00,2026-06-25 00:00:00,77,19.39,4.64,Bafia,Centre,4.75,11.23
2,3,2020-01-03,3,32.0,2026-03-19 00:00:00,25.0,2026-07-31 00:00:00,2026-02-19 00:00:00,2026-01-25 00:00:00,2020-01-03 06:23:00,...,0.0,2026-09-08 00:00:00,2026-09-25 00:00:00,65,2026-03-20 00:00:00,4.78,Bafia,Centre,4.75,11.23
3,4,2020-01-04,3,2026-05-31 00:00:00,2026-01-19 00:00:00,2026-07-24 00:00:00,32.3,2026-07-20 00:00:00,2026-04-25 00:00:00,2020-01-04 06:24:00,...,0.0,2026-02-11 00:00:00,2026-02-20 00:00:00,106,20.48,4.59,Bafia,Centre,4.75,11.23
4,5,2020-01-05,3,31.9,2026-03-19 00:00:00,2026-08-24 00:00:00,33.1,2026-03-20 00:00:00,2026-09-25 00:00:00,2020-01-05 06:24:00,...,0.0,2026-03-11 00:00:00,2026-08-23 00:00:00,100,19.98,4.39,Bafia,Centre,4.75,11.23


In [26]:
#types des variables
df.dtypes

id                                      int64
time                           datetime64[ns]
weather_code                            int64
temperature_2m_max                     object
temperature_2m_min                     object
temperature_2m_mean                    object
apparent_temperature_max               object
apparent_temperature_min               object
apparent_temperature_mean              object
sunrise                        datetime64[ns]
sunset                         datetime64[ns]
daylight_duration                     float64
sunshine_duration                      object
precipitation_sum                      object
rain_sum                               object
snowfall_sum                          float64
precipitation_hours                   float64
wind_speed_10m_max                     object
wind_gusts_10m_max                     object
wind_direction_10m_dominant             int64
shortwave_radiation_sum                object
et0_fao_evapotranspiration        

            ON CONSTATE QUE LES VARRIABLES temperature_2m_max	temperature_2m_min	temperature_2m_mean	apparent_temperature_max	apparent_temperature_min	apparent_temperature_mean	s
            SONT CORROMPU A LA PLACE DES TEMPÉRATURES COMME VALEURS NOUS AVONS LES DATES .

            POUR Y REMEDIER NOUS ALLONS RÉCUPÉRER CES VALEURS DIRECTEMENT SUR OPEN MÉTÉO  PUIS  LES REMPLACER DANS LE DATASET 

                          RECUPERATION DES DONNÉES 

In [3]:

import requests
import pandas as pd
import numpy as np
import time
import os
from datetime import datetime

#  CONFIGURATION 
START_DATE  = "2020-01-01"
END_DATE    = "2025-12-20"
OUTPUT_FILE = "temperatures_openmeteo.csv"

# API Open-Meteo Historical Weather 
API_URL = "https://archive-api.open-meteo.com/v1/archive"

# Variables demandées 
VARIABLES = ",".join([
    "temperature_2m_max",
    "temperature_2m_min",
    "temperature_2m_mean",
    "apparent_temperature_max",
    "apparent_temperature_min",
    "apparent_temperature_mean",
])

#  40 VILLES 
CITIES = [
    {"city": "Abong-Mbang",  "latitude":  3.98, "longitude": 13.17, "region": "Est"},
    {"city": "Akonolinga",   "latitude":  3.77, "longitude": 12.25, "region": "Centre"},
    {"city": "Ambam",        "latitude":  2.38, "longitude": 11.28, "region": "Sud"},
    {"city": "Bafia",        "latitude":  4.75, "longitude": 11.23, "region": "Centre"},
    {"city": "Bafoussam",    "latitude":  5.48, "longitude": 10.42, "region": "Ouest"},
    {"city": "Bamenda",      "latitude":  5.96, "longitude": 10.15, "region": "Nord-Ouest"},
    {"city": "Batouri",      "latitude":  4.43, "longitude": 14.36, "region": "Est"},
    {"city": "Bertoua",      "latitude":  4.57, "longitude": 13.68, "region": "Est"},
    {"city": "Buea",         "latitude":  4.15, "longitude":  9.24, "region": "Sud-Ouest"},
    {"city": "Douala",       "latitude":  4.05, "longitude":  9.70, "region": "Littoral"},
    {"city": "Dschang",      "latitude":  5.44, "longitude": 10.07, "region": "Ouest"},
    {"city": "Ebolowa",      "latitude":  2.91, "longitude": 11.15, "region": "Sud"},
    {"city": "Edea",         "latitude":  3.80, "longitude": 10.13, "region": "Littoral"},
    {"city": "Foumban",      "latitude":  5.72, "longitude": 10.90, "region": "Ouest"},
    {"city": "Garoua",       "latitude":  9.30, "longitude": 13.40, "region": "Nord"},
    {"city": "Guider",       "latitude":  9.93, "longitude": 13.94, "region": "Nord"},
    {"city": "Kousseri",     "latitude": 12.08, "longitude": 15.03, "region": "Extreme-Nord"},
    {"city": "Kribi",        "latitude":  2.95, "longitude":  9.91, "region": "Sud"},
    {"city": "Kumba",        "latitude":  4.63, "longitude":  9.44, "region": "Sud-Ouest"},
    {"city": "Kumbo",        "latitude":  6.20, "longitude": 10.68, "region": "Nord-Ouest"},
    {"city": "Limbe",        "latitude":  4.02, "longitude":  9.20, "region": "Sud-Ouest"},
    {"city": "Loum",         "latitude":  4.71, "longitude":  9.73, "region": "Littoral"},
    {"city": "Mamfe",        "latitude":  5.75, "longitude":  9.31, "region": "Sud-Ouest"},
    {"city": "Maroua",       "latitude": 10.59, "longitude": 14.32, "region": "Extreme-Nord"},
    {"city": "Mbalmayo",     "latitude":  3.52, "longitude": 11.50, "region": "Centre"},
    {"city": "Mbengwi",      "latitude":  6.01, "longitude": 10.00, "region": "Nord-Ouest"},
    {"city": "Mbouda",       "latitude":  5.62, "longitude": 10.25, "region": "Ouest"},
    {"city": "Meiganga",     "latitude":  6.52, "longitude": 14.29, "region": "Adamaoua"},
    {"city": "Mokolo",       "latitude": 10.74, "longitude": 13.80, "region": "Extreme-Nord"},
    {"city": "Ngaoundere",   "latitude":  7.32, "longitude": 13.58, "region": "Adamaoua"},
    {"city": "Nkongsamba",   "latitude":  4.95, "longitude":  9.93, "region": "Littoral"},
    {"city": "Poli",         "latitude":  8.47, "longitude": 13.24, "region": "Nord"},
    {"city": "Sangmelima",   "latitude":  2.93, "longitude": 11.98, "region": "Sud"},
    {"city": "Tibati",       "latitude":  6.46, "longitude": 12.62, "region": "Adamaoua"},
    {"city": "Tignere",      "latitude":  7.37, "longitude": 12.65, "region": "Adamaoua"},
    {"city": "Touboro",      "latitude":  7.76, "longitude": 15.36, "region": "Nord"},
    {"city": "Wum",          "latitude":  6.38, "longitude":  9.95, "region": "Nord-Ouest"},
    {"city": "Yagoua",       "latitude": 10.34, "longitude": 15.23, "region": "Extreme-Nord"},
    {"city": "Yaounde",      "latitude":  3.87, "longitude": 11.52, "region": "Centre"},
    {"city": "Yokadouma",    "latitude":  3.52, "longitude": 15.05, "region": "Est"},
]


#  FONCTION RÉCUPÉRATION UNE VILLE 
def fetch_city(city_info, retries=3):
    city   = city_info["city"]
    lat    = city_info["latitude"]
    lon    = city_info["longitude"]
    region = city_info["region"]

    params = {
        "latitude":   lat,
        "longitude":  lon,
        "daily":      VARIABLES,   
        "start_date": START_DATE,
        "end_date":   END_DATE,
        "timezone":   "Africa/Douala",
    }

    for attempt in range(retries):
        try:
            response = requests.get(API_URL, params=params, timeout=60)
            response.raise_for_status()
            data = response.json()
            break
        except requests.exceptions.RequestException as e:
            if attempt < retries - 1:
                wait = (attempt + 1) * 5
                print(f"  Retry {attempt+1} dans {wait}s...")
                time.sleep(wait)
            else:
                print(f"  Échec : {e}")
                return None

    daily = data.get("daily", {})
    dates = daily.get("time", [])

    if not dates:
        print(f"    Réponse vide")
        return None

    n = len(dates)

    df = pd.DataFrame({
        "time":                      pd.to_datetime(dates),
        "temperature_2m_max":        daily.get("temperature_2m_max",        [np.nan]*n),
        "temperature_2m_min":        daily.get("temperature_2m_min",        [np.nan]*n),
        "temperature_2m_mean":       daily.get("temperature_2m_mean",       [np.nan]*n),
        "apparent_temperature_max":  daily.get("apparent_temperature_max",  [np.nan]*n),
        "apparent_temperature_min":  daily.get("apparent_temperature_min",  [np.nan]*n),
        "apparent_temperature_mean": daily.get("apparent_temperature_mean", [np.nan]*n),
        "city":      city,
        "region":    region,
        "latitude":  lat,
        "longitude": lon,
    })

    return df


# BOUCLE PRINCIPALE 

print("  RÉCUPÉRATION TEMPÉRATURES — OPEN-METEO HISTORICAL")
print(f"  Période  : {START_DATE} → {END_DATE}")
print(f"  Villes   : {len(CITIES)}")
print(f"  Variables: temp max/min/mean + apparent max/min/mean")



all_results = []
errors      = []
start_time  = datetime.now()

for i, city_info in enumerate(CITIES, 1):
    city = city_info["city"]
    pct  = i / len(CITIES) * 100

    print(f"\n[{i:02d}/{len(CITIES)}] {city:<20} ({pct:.0f}%)", end=" ... ")

    df_city = fetch_city(city_info)

    if df_city is not None:
        all_results.append(df_city)
        n_jours  = len(df_city)
        n_valides = df_city["temperature_2m_max"].notna().sum()
        pct_ok   = n_valides / n_jours * 100
        print(f" {n_jours} jours | temp_max valides : {pct_ok:.0f}%")
    else:
        errors.append(city)
        print("  Échec")

    time.sleep(0.5)


#  ASSEMBLAGE ET SAUVEGARDE 

print("  ASSEMBLAGE FINAL")


if not all_results:
    print("Aucune donnée récupérée.")
else:
    df_final = pd.concat(all_results, ignore_index=True)
    df_final = df_final.sort_values(["city", "time"]).reset_index(drop=True)

    print(f"\n  Lignes        : {len(df_final):,}")
    print(f"  Villes        : {df_final['city'].nunique()}")
    print(f"  Période       : {df_final['time'].min().date()} → {df_final['time'].max().date()}")

    print(f"\n  Couverture par variable :")
    for col in ["temperature_2m_max","temperature_2m_min","temperature_2m_mean",
                "apparent_temperature_max","apparent_temperature_min","apparent_temperature_mean"]:
        pct   = df_final[col].notna().mean() * 100
        barre = "█" * int(pct/5) + "░" * (20 - int(pct/5))
        print(f"    {col:<30} [{barre}]  {pct:.1f}%")

    df_final.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
    taille  = os.path.getsize(OUTPUT_FILE) / 1024 / 1024
    elapsed = (datetime.now() - start_time).seconds // 60


    print(f"    FICHIER SAUVEGARDÉ : {OUTPUT_FILE}")
    print(f"      Taille  : {taille:.1f} MB")
    print(f"      Durée   : {elapsed} minutes")


    if errors:
        print(f"\n    Villes en échec : {', '.join(errors)}")

   

  RÉCUPÉRATION TEMPÉRATURES — OPEN-METEO HISTORICAL
  Période  : 2020-01-01 → 2025-12-20
  Villes   : 40
  Variables: temp max/min/mean + apparent max/min/mean

[01/40] Abong-Mbang          (2%) ...  2181 jours | temp_max valides : 100%

[02/40] Akonolinga           (5%) ...  2181 jours | temp_max valides : 100%

[03/40] Ambam                (8%) ...  2181 jours | temp_max valides : 100%

[04/40] Bafia                (10%) ...  2181 jours | temp_max valides : 100%

[05/40] Bafoussam            (12%) ...  2181 jours | temp_max valides : 100%

[06/40] Bamenda              (15%) ...  2181 jours | temp_max valides : 100%

[07/40] Batouri              (18%) ...  2181 jours | temp_max valides : 100%

[08/40] Bertoua              (20%) ...   Retry 1 dans 5s...
  Retry 2 dans 10s...
  Échec : 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=4.57&longitude=13.68&daily=temperature_2m_max%2Ctemperature_2m_min%2Ctemperature_2m_mean%2Capparent_temp

            UNE FOIS LES DONNÉES RÉCUPÉRER FUSIONNONS LES DEUX DATATSETS 

In [7]:

FICHIER_METEO  = "Dataset_complet_Meteo.xlsx"
FICHIER_TEMP   = "temperatures_openmeteo.csv"
FICHIER_SORTIE = "dataset_fusion_temperatures.csv"



print("  VÉRIFICATION DES FICHIERS")


for f in [FICHIER_METEO, FICHIER_TEMP]:
    if not os.path.exists(f):
        raise FileNotFoundError(f" Fichier introuvable : {f}")
    taille = os.path.getsize(f) / 1024 / 1024
    print(f"   {f}  ({taille:.1f} MB)")



print("  CHARGEMENT DATASET MÉTÉO ORIGINAL (.xlsx)")


df_meteo = pd.read_excel(FICHIER_METEO)
df_meteo['time'] = pd.to_datetime(df_meteo['time']).dt.normalize()

print(f"  Lignes    : {len(df_meteo):,}")
print(f"  Colonnes  : {df_meteo.shape[1]}")
print(f"  Villes    : {df_meteo['city'].nunique()}")
print(f"  Période   : {df_meteo['time'].min().date()} → {df_meteo['time'].max().date()}")



print("  CHARGEMENT TEMPÉRATURES OPEN-METEO")


df_temp = pd.read_csv(FICHIER_TEMP, low_memory=False)
df_temp['time'] = pd.to_datetime(df_temp['time']).dt.normalize()

print(f"  Lignes    : {len(df_temp):,}")
print(f"  Villes    : {df_temp['city'].nunique()}")
print(f"  Période   : {df_temp['time'].min().date()} → {df_temp['time'].max().date()}")

# Couverture des températures open-meteo
cols_temp = [
    "temperature_2m_max", "temperature_2m_min", "temperature_2m_mean",
    "apparent_temperature_max", "apparent_temperature_min", "apparent_temperature_mean"
]
print(f"\n  Couverture températures open-meteo :")
for col in cols_temp:
    if col in df_temp.columns:
        pct = df_temp[col].notna().mean() * 100
        print(f"    {col:<35} {pct:.1f}%")




print("  COUVERTURE TEMPÉRATURES DANS LE XLSX (avant remplacement)")


for col in cols_temp:
    if col in df_meteo.columns:
        ser = pd.to_numeric(df_meteo[col], errors='coerce')
        pct = ser.notna().mean() * 100
        print(f"    {col:<35} {pct:.1f}%  ({' corrompue' if pct < 50 else 'ok'})")



# ÉTAPE  DE FUSION
# On supprime les colonnes température corrompues du xlsx
# et on les remplace par celles d'open-meteo via LEFT JOIN
# Les autres colonnes du xlsx (vent, pluie, radiation...) restent intactes


print("  FUSION")


# Étape 1 — Supprimer les colonnes température du xlsx

cols_a_remplacer = [c for c in cols_temp if c in df_meteo.columns]
df_meteo_sans_temp = df_meteo.drop(columns=cols_a_remplacer)
print(f"  Colonnes température supprimées du xlsx : {cols_a_remplacer}")

# Étape 2 — Préparer le fichier températures open-meteo

cols_temp_disponibles = [c for c in cols_temp if c in df_temp.columns]
df_temp_reduit = df_temp[['city', 'time'] + cols_temp_disponibles].copy()
print(f"  Colonnes température open-meteo à ajouter : {cols_temp_disponibles}")

# Étape 3 — LEFT JOIN sur city + time
df_fusion = pd.merge(
    df_meteo_sans_temp,
    df_temp_reduit,
    on=['city', 'time'],
    how='left'
)

print(f"\n  Lignes avant fusion : {len(df_meteo):,}")
print(f"  Lignes après fusion : {len(df_fusion):,}")

# Vérification — aucune perte de lignes
assert len(df_fusion) == len(df_meteo), \
    f"ANOMALIE : {len(df_fusion)} lignes au lieu de {len(df_meteo)}"
print(f" Aucune perte de lignes")

# Vérification doublons
dupes = df_fusion.duplicated(subset=['city', 'time']).sum()
print(f"  Doublons (city+time) : {dupes}  {'YES' if dupes == 0 else 'NO' }")




print("  COUVERTURE TEMPÉRATURES APRÈS FUSION (open-meteo)")


for col in cols_temp_disponibles:
    pct = df_fusion[col].notna().mean() * 100
    barre = "█" * int(pct/5) + "░" * (20 - int(pct/5))
    print(f"    {col:<35} [{barre}]  {pct:.1f}%")



print("  RÉSUMÉ DATASET FINAL")


print(f"\n  Lignes   : {len(df_fusion):,}")
print(f"  Colonnes : {df_fusion.shape[1]}")
print(f"  Villes   : {df_fusion['city'].nunique()}")
print(f"  Régions  : {df_fusion['region'].nunique()}")
print(f"  Période  : {df_fusion['time'].min().date()} → {df_fusion['time'].max().date()}")

print(f"\n  Liste complète des colonnes :")
for i, col in enumerate(df_fusion.columns, 1):
    print(f"    {i:02d}. {col}")




print("  SAUVEGARDE")


df_fusion.to_csv(FICHIER_SORTIE, index=False, encoding='utf-8')

taille = os.path.getsize(FICHIER_SORTIE) / 1024 / 1024
print(f"\n  Fichier sauvegardé : {FICHIER_SORTIE}")
print(f"     Taille   : {taille:.1f} MB")
print(f"     Lignes   : {len(df_fusion):,}")
print(f"     Colonnes : {df_fusion.shape[1]}")



SyntaxError: incomplete input (1012648488.py, line 146)

                                    VÉRIFICATION DU NOUVEAUX DATASET 

In [ ]:
#chargement du dataset
df2 = pd.read_csv('dataset_fusion_temperatures.csv')

In [ ]:
# taille 
df2.shape